# MATRA-X Benchmark Notebook
Alchemy Architects · SIH26099

This notebook provides a reproducible benchmark path for lexical, embedding and hybrid material matching. It is safe to run first on the bundled demo CSV, then on public research data or Kaggle imports.

In [ ]:
!pip -q install pandas numpy scikit-learn rapidfuzz sentence-transformers


In [ ]:
import pandas as pd, numpy as np, re
from pathlib import Path
from rapidfuzz.fuzz import ratio

DATA=Path("data/demo_materials.csv")
df=pd.read_csv(DATA)
df.head()

## Optional public data tracks
- Public SIH26099 material corpus: https://huggingface.co/datasets/sarthak20024/sih26099-cpse-material-codes
- Team-supplied Kaggle candidates are listed in `app/data/SOURCES.md`. Treat them as auxiliary product/entity-linking data, not CPSE ground truth.

In [ ]:
def norm(x):
    return re.sub(r"[^a-z0-9]+"," ",str(x).lower()).strip()

df["norm_desc"]=df["description"].map(norm)

# Pair examples: same canonical vs hard-negative
pairs=[]
for _,a in df.iterrows():
    for _,b in df.iterrows():
        if a["material_code"] < b["material_code"] and (a["canonical_hint"]==b["canonical_hint"] or "HARD" in str(a["canonical_hint"])):
            pairs.append((a,b,1 if a["canonical_hint"]==b["canonical_hint"] else 0))
pair_df=pd.DataFrame([{**{f"a_{k}":v for k,v in a.items()},**{f"b_{k}":v for k,v in b.items()},"label":y} for a,b,y in pairs])
pair_df.head()

In [ ]:
# Embedding benchmark
from sentence_transformers import SentenceTransformer
model=SentenceTransformer("all-MiniLM-L6-v2")
emb=model.encode(df["norm_desc"].tolist(), normalize_embeddings=True)

# Replace this model with a domain-appropriate embedding model for final experiments.


In [ ]:
# Hybrid scoring example
ATTR=["category","uom","size","dimension","material_grade","standard"]
def hybrid(a,b):
    sim=float(np.dot(emb[a.name],emb[b.name]))
    am=sum(bool(a[k]) and bool(b[k]) and str(a[k]).strip().lower()==str(b[k]).strip().lower() for k in ATTR)/len(ATTR)
    conflicts=[k for k in ATTR if a[k] and b[k] and str(a[k]).strip().lower()!=str(b[k]).strip().lower()]
    score=.55*sim+.45*am
    if conflicts: score*=.6
    return score, conflicts

examples=[]
for _,a in df.iterrows():
    for _,b in df.iterrows():
        if a["material_code"]<b["material_code"]:
            s,c=hybrid(a,b)
            if s>.55: examples.append((a["material_code"],b["material_code"],round(s,3),c))
pd.DataFrame(examples,columns=["a","b","score","conflicts"]).sort_values("score",ascending=False).head(20)

## Final evaluation
For the real benchmark, report: precision, recall, F1, top-k candidate recall, false-merge rate, critical-conflict recall, calibration/error, reviewer acceptance rate, and review workload. Keep train/test entities disjoint to avoid leakage.